In [1]:
import os
import pickle
from pathlib import Path

# WSL crash workaround: avoid TensorFlow/Triton code paths in this notebook.
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Do not set CUDA_LAUNCH_BLOCKING=1 during training (slows GPU heavily).

# True = smaller/faster run for tuning; False = larger production run
FAST_TRAIN = True

import evaluate
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainerCallback,
    TrainingArguments,
)

PROJECT_DIR = Path.cwd()

BASE_MODEL_DIR = PROJECT_DIR / "models" / "DACTYL"
BASE_WEIGHT_FILE = BASE_MODEL_DIR / "model.safetensors"

with open("pickles/train_samples.pkl", "rb") as f:
    train_samples = pickle.load(f)
with open("pickles/validation_samples.pkl", "rb") as f:
    val_samples = pickle.load(f)
with open("pickles/test_samples.pkl", "rb") as f:
    test_samples = pickle.load(f)

print("train_samples:", len(train_samples))
print("val_samples:", len(val_samples))
print("test_samples:", len(test_samples))

train_samples: 1137319
val_samples: 71082
test_samples: 213248


In [2]:
print(torch.backends.cuda.is_built())

True


In [3]:
print("torch:", torch.__version__, "| cuda build:", torch.version.cuda)
print("cuda.is_available:", torch.cuda.is_available())


def cuda_kernels_work() -> bool:
    """RTX 50xx (sm_120) needs PyTorch cu128+; cu121 reports cuda=True but kernels fail."""
    if not torch.cuda.is_available():
        return False
    try:
        torch.zeros(1, device="cuda").add_(1)
        return True
    except RuntimeError:
        return False


if cuda_kernels_work():
    device = torch.device("cuda")
    USE_BF16 = True
    USE_FP16 = False
    print("GPU:", torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
else:
    device = torch.device("cpu")
    USE_BF16 = False
    USE_FP16 = False
    if torch.cuda.is_available():
        print(
            "CUDA device visible but PyTorch cannot run kernels (common on RTX 50xx / sm_120).\n"
            "Upgrade PyTorch, then restart the kernel:\n"
            "  pip install --upgrade torch --index-url https://download.pytorch.org/whl/cu128\n"
            "Or nightly: pip install --pre torch --index-url https://download.pytorch.org/whl/nightly/cu128\n"
            "Training will use CPU until then (slow but works)."
        )
    else:
        print("No CUDA — training on CPU.")

print("device:", device, "| bf16:", USE_BF16, "| fp16:", USE_FP16)

torch: 2.11.0+cu128 | cuda build: 12.8
cuda.is_available: True
GPU: NVIDIA GeForce RTX 5060 (12, 0)
device: cuda | bf16: True | fp16: False


In [4]:
if not BASE_MODEL_DIR.exists() or not BASE_WEIGHT_FILE.exists():
    raise FileNotFoundError(
        f"Missing local model at {BASE_MODEL_DIR}.\n"
        "Download once with:\n"
        "  huggingface-cli download ShantanuT01/dactyl-ai-text-detector --local-dir models/DACTYL"
    )

try:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, use_fast=True)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, use_fast=False)
base_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_DIR)

print("Loaded from:", BASE_MODEL_DIR)
print("Tokenizer:", tokenizer.__class__.__name__)
print("Base model:", base_model.__class__.__name__)
print("device (from earlier cell):", device)

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Loaded from: E:\_project\human-ai-mixed-data-generator\models\DACTYL
Tokenizer: DebertaV2Tokenizer
Base model: DebertaV2ForSequenceClassification
device (from earlier cell): cuda


In [5]:
class DebertaTokenClassifier(nn.Module):
    """Token-level head on top of frozen DeBERTa encoder."""

    def __init__(self, backbone_seq_model, num_labels=2, dropout=0.1):
        super().__init__()
        self.deberta = backbone_seq_model.deberta
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(backbone_seq_model.config.hidden_size, num_labels)
        self.num_labels = num_labels

        # Freeze backbone parameters
        for p in self.deberta.parameters():
            p.requires_grad = False

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        logits = self.classifier(sequence_output)  # [batch, seq, num_labels]

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return {"loss": loss, "logits": logits}


model = DebertaTokenClassifier(base_model, num_labels=2).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,}")
print(f"Total params: {total:,}")
print(f"Trainable ratio: {100 * trainable / total:.4f}%")

Trainable params: 2,050
Total params: 434,014,210
Trainable ratio: 0.0005%


In [6]:
from training_data import (
    CachedTokenDataset,
    make_token_collator,
    pretokenize_split,
    training_limits,
)

# Set True to rebuild pickles/token_cache/*.pkl after changing limits or MAX_LENGTH
REBUILD_CACHE = False

cfg = training_limits(FAST_TRAIN)
globals().update(cfg)

train_data = train_samples[:TRAIN_LIMIT] if TRAIN_LIMIT else train_samples
val_data = val_samples[:VAL_LIMIT] if VAL_LIMIT else val_samples
test_data = test_samples[:TEST_LIMIT] if TEST_LIMIT else test_samples

TOKEN_CACHE_DIR = PROJECT_DIR / "pickles" / "token_cache"

def token_cache_path(split, n_samples):
    return TOKEN_CACHE_DIR / f"{split}_n{n_samples}_len{MAX_LENGTH}.pkl"

print(
    f"FAST_TRAIN={FAST_TRAIN} | train={len(train_data):,} val={len(val_data):,} "
    f"| MAX_LENGTH={MAX_LENGTH} | pretokenize_batch={PRETOKENIZE_BATCH_SIZE}"
)
if not FAST_TRAIN:
    print("Tip: FAST_TRAIN=True in cell 1 is much faster for experiments (100k samples).")

token_collator = make_token_collator(tokenizer)

train_rows = pretokenize_split(
    "train", train_data, tokenizer, MAX_LENGTH,
    token_cache_path("train", len(train_data)),
    batch_size=PRETOKENIZE_BATCH_SIZE, rebuild_cache=REBUILD_CACHE,
)
val_rows = pretokenize_split(
    "val", val_data, tokenizer, MAX_LENGTH,
    token_cache_path("val", len(val_data)),
    batch_size=PRETOKENIZE_BATCH_SIZE, rebuild_cache=REBUILD_CACHE,
)
test_rows = pretokenize_split(
    "test", test_data, tokenizer, MAX_LENGTH,
    token_cache_path("test", len(test_data)),
    batch_size=PRETOKENIZE_BATCH_SIZE, rebuild_cache=REBUILD_CACHE,
)

train_ds = CachedTokenDataset(train_rows)
val_ds = CachedTokenDataset(val_rows)
test_ds = CachedTokenDataset(test_rows)

sample = train_ds[0]
eff_batch = TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS
print(f"effective_batch={eff_batch}")
print("sample shapes:", sample["input_ids"].shape, sample["labels"].shape)
print("dataset sizes:", len(train_ds), len(val_ds), len(test_ds))

FAST_TRAIN=True | train=100,000 val=5,000 | MAX_LENGTH=128 | pretokenize_batch=1024
[train] Loading cache (train_n100000_len128.pkl)
[val] Loading cache (val_n5000_len128.pkl)
[test] Loading cache (test_n10000_len128.pkl)
effective_batch=64
sample shapes: torch.Size([42]) torch.Size([42])
dataset sizes: 100000 5000 10000


In [7]:
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")


In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    # Flatten while ignoring masked labels
    labels_flat = labels.reshape(-1)
    preds_flat = preds.reshape(-1)
    keep = labels_flat != -100

    labels_kept = labels_flat[keep]
    preds_kept = preds_flat[keep]

    if len(labels_kept) == 0:
        return {"token_accuracy": 0.0, "token_f1": 0.0}

    acc = metric_acc.compute(predictions=preds_kept, references=labels_kept)["accuracy"]
    f1 = metric_f1.compute(predictions=preds_kept, references=labels_kept, average="binary")["f1"]
    return {"token_accuracy": acc, "token_f1": f1}

In [9]:
class StepProgressCallback(TrainerCallback):
    """Print clear step/epoch progress during training."""

    def on_train_begin(self, args, state, control, **kwargs):
        print(f"Training started | epochs={args.num_train_epochs} | max_steps={state.max_steps}")

    def on_epoch_begin(self, args, state, control, **kwargs):
        epoch = int(state.epoch) + 1
        print(f"\n=== Epoch {epoch}/{int(args.num_train_epochs)} ===")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step = state.global_step
        max_steps = state.max_steps or 1
        pct = 100.0 * step / max_steps
        msg = [f"step {step}/{max_steps} ({pct:.1f}%)"]
        if "loss" in logs:
            msg.append(f"loss={logs['loss']:.4f}")
        if "learning_rate" in logs:
            msg.append(f"lr={logs['learning_rate']:.2e}")
        if "token_accuracy" in logs:
            msg.append(f"token_acc={logs['token_accuracy']:.4f}")
        if "token_f1" in logs:
            msg.append(f"token_f1={logs['token_f1']:.4f}")
        print(" | ".join(msg), flush=True)

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            parts = [f"{k}={v:.4f}" for k, v in metrics.items() if isinstance(v, (int, float))]
            print(f"[eval @ step {state.global_step}] " + ", ".join(parts), flush=True)

    def on_train_end(self, args, state, control, **kwargs):
        print(f"Training finished at step {state.global_step}", flush=True)


OUTPUT_DIR = PROJECT_DIR / "output" / "dactyl_token_finetuned"

_train_kw = dict(
    output_dir=str(OUTPUT_DIR),
    learning_rate=2e-4,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    eval_strategy=EVAL_STRATEGY,
    save_strategy=SAVE_STRATEGY,
    load_best_model_at_end=True,
    metric_for_best_model="token_f1",
    greater_is_better=True,
    logging_steps=10,
    logging_first_step=True,
    disable_tqdm=False,
    dataloader_num_workers=0,  # Windows+Jupyter: avoid __main__ pickle errors
    dataloader_pin_memory=device.type == "cuda",
    fp16=USE_FP16,
    bf16=USE_BF16,
    tf32=device.type == "cuda",
    optim="adamw_torch",
    report_to="none",
)
if EVAL_STEPS is not None:
    _train_kw["eval_steps"] = EVAL_STEPS
if SAVE_STEPS is not None:
    _train_kw["save_steps"] = SAVE_STEPS
training_args = TrainingArguments(**_train_kw)
print(f"Training config: batch={TRAIN_BATCH_SIZE} accum={GRAD_ACCUM_STEPS} epochs={NUM_EPOCHS}")

progress_callback = StepProgressCallback()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=token_collator,
    compute_metrics=compute_metrics,
    callbacks=[progress_callback],
)


Training config: batch=16 accum=4 epochs=1


In [10]:
# Run training (watch step logs + tqdm bar)
try:
    train_result = trainer.train()
    print("Final train metrics:", train_result.metrics)
except Exception as e:
    print("Training failed:", repr(e))
    raise

Training started | epochs=1 | max_steps=1563

=== Epoch 1/1 ===


Step,Training Loss,Validation Loss,Token Accuracy,Token F1
200,0.700488,0.687717,0.560717,0.000012
400,0.698327,0.691524,0.560780,0.000000
600,0.684514,0.688024,0.560796,0.000000
800,0.693741,0.686814,0.560796,0.000000
1000,0.687270,0.685476,0.560796,0.000000
1200,0.680814,0.686037,0.560796,0.000000
1400,0.688343,0.687305,0.560796,0.000000


step 1/1563 (0.1%) | loss=0.7507 | lr=2.00e-04
step 10/1563 (0.6%) | loss=0.7026 | lr=1.99e-04
step 20/1563 (1.3%) | loss=0.6810 | lr=1.98e-04
step 30/1563 (1.9%) | loss=0.7032 | lr=1.96e-04
step 40/1563 (2.6%) | loss=0.7109 | lr=1.95e-04
step 50/1563 (3.2%) | loss=0.7039 | lr=1.94e-04
step 60/1563 (3.8%) | loss=0.6918 | lr=1.92e-04
step 70/1563 (4.5%) | loss=0.6784 | lr=1.91e-04
step 80/1563 (5.1%) | loss=0.6954 | lr=1.90e-04
step 90/1563 (5.8%) | loss=0.6974 | lr=1.89e-04
step 100/1563 (6.4%) | loss=0.7062 | lr=1.87e-04
step 110/1563 (7.0%) | loss=0.6994 | lr=1.86e-04
step 120/1563 (7.7%) | loss=0.6888 | lr=1.85e-04
step 130/1563 (8.3%) | loss=0.7066 | lr=1.83e-04
step 140/1563 (9.0%) | loss=0.7007 | lr=1.82e-04
step 150/1563 (9.6%) | loss=0.6934 | lr=1.81e-04
step 160/1563 (10.2%) | loss=0.6930 | lr=1.80e-04
step 170/1563 (10.9%) | loss=0.6987 | lr=1.78e-04
step 180/1563 (11.5%) | loss=0.7022 | lr=1.77e-04
step 190/1563 (12.2%) | loss=0.6972 | lr=1.76e-04
step 200/1563 (12.8%) | los

In [ ]:
import json

test_metrics = trainer.evaluate(test_ds)
print("Test metrics:", test_metrics)

best_dir = OUTPUT_DIR / "best"
best_dir.mkdir(parents=True, exist_ok=True)

# Save tokenizer
tokenizer.save_pretrained(best_dir)

# Save custom token-classifier weights + minimal metadata
weights_path = best_dir / "token_classifier_state.pt"
torch.save(model.state_dict(), weights_path)

meta = {
    "base_model_dir": str(BASE_MODEL_DIR),
    "num_labels": 2,
    "max_length": MAX_LENGTH,
    "model_class": "DebertaTokenClassifier",
}
with open(best_dir / "token_classifier_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

if not weights_path.exists():
    raise RuntimeError(f"Weight save failed: {weights_path} not found")

print("Saved token-level model to", best_dir)
print("Weights:", weights_path)
print("Meta:", best_dir / "token_classifier_meta.json")

# ---- Optional: export as a HuggingFace token-classification model bundle ----
hf_export_dir = OUTPUT_DIR / "best_hf"
hf_export_dir.mkdir(parents=True, exist_ok=True)

hf_config = AutoConfig.from_pretrained(BASE_MODEL_DIR)
hf_config.num_labels = 2
hf_config.id2label = {0: "human", 1: "ai"}
hf_config.label2id = {"human": 0, "ai": 1}

# Base DACTYL is sequence-classification (1 logit). Do not from_pretrained into a 2-label head.
hf_model = AutoModelForTokenClassification.from_config(hf_config)
hf_model.deberta.load_state_dict(model.deberta.state_dict(), strict=True)
hf_model.classifier.load_state_dict(model.classifier.state_dict(), strict=True)

hf_model.save_pretrained(hf_export_dir)
tokenizer.save_pretrained(hf_export_dir)

print("Exported HuggingFace model bundle to", hf_export_dir)
print("Includes config.json + model.safetensors + tokenizer files")

In [ ]:
# load fine-tuned token-level model from local directory

import json
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SAVE_DIR = PROJECT_DIR / "output" / "dactyl_token_finetuned" / "best"
WEIGHT_FILE = SAVE_DIR / "token_classifier_state.pt"
META_FILE = SAVE_DIR / "token_classifier_meta.json"

if not SAVE_DIR.exists():
    raise FileNotFoundError(f"Missing model directory: {SAVE_DIR}")
if not WEIGHT_FILE.exists() or not META_FILE.exists():
    raise FileNotFoundError(
        f"Missing custom token model files in {SAVE_DIR}.\n"
        "Run the training+save cell first."
    )

with open(META_FILE, "r") as f:
    meta = json.load(f)

base_model_dir = Path(meta["base_model_dir"])
num_labels = int(meta.get("num_labels", 2))

# Guard against stale/wrong metadata from earlier saves
if not (base_model_dir / "config.json").exists():
    print(f"Warning: invalid base_model_dir in meta: {base_model_dir}")
    base_model_dir = PROJECT_DIR / "models" / "DACTYL"

if not WEIGHT_FILE.exists():
    raise FileNotFoundError(f"Missing token head weights: {WEIGHT_FILE}. Re-run save cell.")

tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
base_for_load = AutoModelForSequenceClassification.from_pretrained(base_model_dir)
best_model = DebertaTokenClassifier(base_for_load, num_labels=num_labels)

state_dict = torch.load(WEIGHT_FILE, map_location="cpu")
best_model.load_state_dict(state_dict)

device = torch.device("cuda") if cuda_kernels_work() else torch.device("cpu")
best_model.to(device)
best_model.eval()

print("Loaded token-level model from", SAVE_DIR)
print("Using backbone:", base_model_dir)

In [ ]:
best_model.eval()

In [ ]:
# Inspect new classifier
model.classifier

In [ ]:
# Load exported HuggingFace token model (config.json + safetensors)
HF_EXPORT_DIR = OUTPUT_DIR / "best_hf"

hf_tokenizer = AutoTokenizer.from_pretrained(HF_EXPORT_DIR)
hf_token_model = AutoModelForTokenClassification.from_pretrained(HF_EXPORT_DIR)

hf_token_model.to(device)
hf_token_model.eval()
print("Loaded HF-exported token model from", HF_EXPORT_DIR)